# Egg Sex Classification — YOLOv5 5-Fold Cross-Validation

Fine-tunes a pretrained YOLOv5s-cls model for binary sex classification of LSCI egg images. Data must be pre-processed by `1_data_preprocessing.ipynb` before running this notebook.

**Reference:** [Roboflow YOLOv5 Classification Notebook](https://github.com/roboflow/notebooks/blob/main/notebooks/train-yolov5-classification-on-custom-data.ipynb)

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pickle
import random
import shutil
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter
import torchvision.transforms as transforms
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import KFold

## 2. Configuration

In [3]:
RANDOM_SEED  = 1839
K_FOLDS      = 5
IMG_SIZE     = 640
EPOCHS       = 300
BATCH_SIZE   = 64
LR           = 1e-4
OPTIMIZER    = 'Adam'
DROPOUT     = 0.1
MODEL_WEIGHT = 'yolov5s-cls.pt'

# Path to pre-processed pickle files (output of 1_data_preprocessing.ipynb)
DATASET_PATH = '/content/drive/MyDrive/CS 163/egg_sex_zoomed/full_640_cv/data'

random.seed(RANDOM_SEED)

## 3. Dataset and Preprocessing

In [4]:
def unsharp_mask(img, strength=0.95):
    """Sharpen a PIL image using an unsharp mask filter."""
    blurred    = img.filter(ImageFilter.GaussianBlur(radius=3.0))
    img_np     = np.array(img,     dtype=np.float32)
    blurred_np = np.array(blurred, dtype=np.float32)
    sharpened  = np.clip(img_np + strength * (img_np - blurred_np), 0, 255).astype(np.uint8)
    return Image.fromarray(sharpened)


class EggDataset(Dataset):
    """Loads all pickle files from a directory into memory."""
    def __init__(self, pickle_dir):
        self.data = []
        self.ids  = []
        for fname in sorted(os.listdir(pickle_dir)):
            if fname.endswith('.pickle') and not fname.startswith('.'):
                with open(os.path.join(pickle_dir, fname), 'rb') as f:
                    self.data.extend(pickle.load(f))
        self.ids = [dp['additional_info']['id'] for dp in self.data]

    def __len__(self):
        return len(self.data)


def cross_validation_splits(dataset, k_folds=5, seed=42):
    """K-fold split by egg ID. The held-out set for each fold is separated
    by developmental stage: stage 19 (HH19) → val, stage 25 (HH25) → test.
    Returns a list of (train, val_hh19, test_hh25) tuples."""
    unique_ids = sorted(set(dataset.ids))
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=seed)
    splits = []
    for train_idx, held_idx in kf.split(unique_ids):
        train_ids  = {unique_ids[i] for i in train_idx}
        held_ids   = {unique_ids[i] for i in held_idx}
        train = [dp for dp in dataset.data if dp['additional_info']['id'] in train_ids]
        held  = [dp for dp in dataset.data if dp['additional_info']['id'] in held_ids]
        val   = [dp for dp in held if dp['additional_info']['stage'] == 19]   # HH19
        test  = [dp for dp in held if dp['additional_info']['stage'] == 25]   # HH25
        splits.append((train, val, test))
    return splits


dataset = EggDataset(DATASET_PATH)
splits  = cross_validation_splits(dataset, k_folds=K_FOLDS, seed=RANDOM_SEED)

hh19_n = sum(1 for dp in dataset.data if dp['additional_info']['stage'] == 19)
hh25_n = sum(1 for dp in dataset.data if dp['additional_info']['stage'] == 25)
print(f'Dataset: {len(dataset)} images, {len(set(dataset.ids))} unique egg IDs')
print(f'  HH19 (val):  {hh19_n} images')
print(f'  HH25 (test): {hh25_n} images')
for i, (tr, va, te) in enumerate(splits):
    print(f'  Fold {i+1}: {len(tr)} train | {len(va)} val (HH19) | {len(te)} test (HH25)')

Dataset: 2773 images, 1251 unique egg IDs
  HH19 (val):  1553 images
  HH25 (test): 1220 images
  Fold 1: 2204 train | 319 val (HH19) | 250 test (HH25)
  Fold 2: 2230 train | 307 val (HH19) | 236 test (HH25)
  Fold 3: 2227 train | 317 val (HH19) | 229 test (HH25)
  Fold 4: 2230 train | 296 val (HH19) | 247 test (HH25)
  Fold 5: 2201 train | 314 val (HH19) | 258 test (HH25)


In [7]:
for i, (tr, va, te) in enumerate(splits):
    tr_ids = sorted(set(dp['additional_info']['id'] for dp in tr))
    va_ids = sorted(set(dp['additional_info']['id'] for dp in va))
    te_ids = sorted(set(dp['additional_info']['id'] for dp in te))
    print(f"Fold {i+1}: train={tr_ids[:3]}... val={va_ids[:3]}... test={te_ids[:3]}...")

Fold 1: train=[1, 2, 4]... val=[8, 11, 19]... test=[8, 11, 19]...
Fold 2: train=[1, 5, 7]... val=[2, 4, 9]... test=[9, 16, 37]...
Fold 3: train=[2, 4, 5]... val=[1, 21, 24]... test=[1, 24, 30]...
Fold 4: train=[1, 2, 4]... val=[10, 27, 44]... test=[13, 27, 44]...
Fold 5: train=[1, 2, 4]... val=[5, 7, 17]... test=[5, 7, 12]...


## 4. Build YOLOv5 Directory Structure

YOLOv5 expects images organised as:
```
datasets/fold{N}/
  train/male/   train/female/
  val/male/     val/female/
  test/male/    test/female/
```

In [8]:
LABEL_MAP = {0: 'male', 1: 'female'}
BASE_DATASETS = '/content/datasets'


def save_fold_to_disk(fold_index, train_data, val_data, test_data):
    """Write processed images for one fold to the YOLOv5 directory layout.
    val  → HH19 held-out eggs (Day 3 evaluation)
    test → HH25 held-out eggs (Day 4 evaluation)
    """
    base = os.path.join(BASE_DATASETS, f'fold{fold_index + 1}')
    if os.path.exists(base):
        shutil.rmtree(base)  # clear old data

    for split_name, split_data in [('train', train_data), ('val', val_data), ('test', test_data)]:
        for label_name in LABEL_MAP.values():
            Path(os.path.join(base, split_name, label_name)).mkdir(parents=True, exist_ok=True)

        for dp in split_data:
            image    = unsharp_mask(dp['image'])
            label    = LABEL_MAP[dp['label']]
            img_id   = dp['additional_info']['id']
            out_dir  = os.path.join(base, split_name, label)
            out_path = os.path.join(out_dir, f'egg_{img_id}_{id(dp)}.png')
            image.save(out_path)

    print(f'  Fold {fold_index + 1}: {len(train_data)} train | '
          f'{len(val_data)} val (HH19) | {len(test_data)} test (HH25)')


print('Writing fold images to disk...')
for fold_idx, (train_data, val_data, test_data) in enumerate(splits):
    save_fold_to_disk(fold_idx, train_data, val_data, test_data)
print('Done.')


Writing fold images to disk...
  Fold 1: 2204 train | 319 val (HH19) | 250 test (HH25)
  Fold 2: 2230 train | 307 val (HH19) | 236 test (HH25)
  Fold 3: 2227 train | 317 val (HH19) | 229 test (HH25)
  Fold 4: 2230 train | 296 val (HH19) | 247 test (HH25)
  Fold 5: 2201 train | 314 val (HH19) | 258 test (HH25)
Done.


## 5. Install YOLOv5

In [9]:
%cd /content

/content


In [10]:
HOME = os.getcwd()
!git clone https://github.com/anika24/yolov5
%cd yolov5
%pip install -qr requirements.txt
import utils; utils.notebook_init()

YOLOv5 🚀 6e924a5 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)


Setup complete ✅ (12 CPUs, 83.5 GB RAM, 46.7/112.6 GB disk)


<module 'IPython.display' from '/usr/local/lib/python3.12/dist-packages/IPython/display.py'>

In [11]:
# Download YOLOv5 classification weights
from utils.downloads import attempt_download
attempt_download(f'weights/{MODEL_WEIGHT}')

100%|██████████| 10.5M/10.5M [00:00<00:00, 39.9MB/s]



'weights/yolov5s-cls.pt'

In [12]:
%cd /content/yolov5
!pip install -q "albumentations==1.3.1"

/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 12.9 MB/s eta 0:00:00


## 6. Train All Folds

Key settings per fold:
- `--patience 15` — early stopping (no improvement for 15 epochs)
- `--freeze 10` — backbone (layers 0–9) frozen; only the classifier head is trained
- Dropout p=0.25 and `A.Rotate(limit=15)` are applied via the source patches above

Each fold's best checkpoint → `runs/train-cls/fold{N}/weights/best.pt`

In [11]:
!cd /content/yolov5 && python -u classify/train.py --model yolov5s-cls.pt --data /content/datasets/fold1/ --imgsz 640 --epochs 300 --lr0 1e-4 --batch-size 32 --optimizer Adam --dropout 0.1 --project /content/runs/train-cls --name fold1

2026-04-19 01:26:50.551907: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776562010.574175    8180 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776562010.581571    8180 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776562010.600663    8180 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776562010.600687    8180 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776562010.600690    8180 computation_placer.cc:177] computation placer alr

In [12]:
!cd /content/yolov5 && python -u classify/train.py --model yolov5s-cls.pt --data /content/datasets/fold2/ --imgsz 640 --epochs 300 --lr0 1e-4 --batch-size 32 --optimizer Adam --dropout 0.1 --project /content/runs/train-cls --name fold2

2026-04-19 01:39:51.171406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776562791.194014   11826 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776562791.201529   11826 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776562791.220636   11826 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776562791.220664   11826 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776562791.220667   11826 computation_placer.cc:177] computation placer alr

In [13]:
!cd /content/yolov5 && python -u classify/train.py --model yolov5s-cls.pt --data /content/datasets/fold3/ --imgsz 640 --epochs 300 --lr0 1e-4 --batch-size 32 --optimizer Adam --dropout 0.1 --project /content/runs/train-cls --name fold3

2026-04-19 01:54:01.135228: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776563641.158894   15759 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776563641.166385   15759 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776563641.186226   15759 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776563641.186255   15759 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776563641.186258   15759 computation_placer.cc:177] computation placer alr

In [13]:
!cd /content/yolov5 && python -u classify/train.py --model yolov5s-cls.pt --data /content/datasets/fold4/ --imgsz 640 --epochs 300 --lr0 1e-4 --batch-size 32 --optimizer Adam --dropout 0.1 --project /content/runs/train-cls --name fold4

2026-04-19 03:28:26.186083: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776569306.208198    8069 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776569306.215516    8069 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776569306.234168    8069 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776569306.234192    8069 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776569306.234195    8069 computation_placer.cc:177] computation placer alr

In [14]:
!cd /content/yolov5 && python -u classify/train.py --model yolov5s-cls.pt --data /content/datasets/fold5/ --imgsz 640 --epochs 300 --lr0 1e-4 --batch-size 32 --optimizer Adam --dropout 0.1 --project /content/runs/train-cls --name fold5

2026-04-19 03:43:39.947979: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776570219.970478   12285 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776570219.977949   12285 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776570219.996990   12285 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776570219.997015   12285 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776570219.997018   12285 computation_placer.cc:177] computation placer alr